In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, BooleanType
from datetime import date

existing_data = [
    # customer_key, customer_id, full_name,         email,                    tier,    address,           start_date,              end_date,   is_current
    ("KEY-001", "C001", "Alice Johnson",  "alice@email.com",  "Gold",   "123 Main St",    date(2022, 1, 1),  None,              True),
    ("KEY-002", "C002", "Bob Smith",      "bob@email.com",    "Silver", "456 Oak Ave",    date(2022, 3, 15), None,              True),
    ("KEY-003", "C003", "Carol White",    "carol@email.com",  "Bronze", "789 Pine Rd",    date(2021, 6, 1),  None,              True),
    ("KEY-004", "C004", "David Lee",      "david@email.com",  "Gold",   "321 Elm St",     date(2023, 2, 1),  None,              True),
    # C002 had an old address — this is the closed historical version
    ("KEY-OLD", "C002", "Bob Smith",      "bob@email.com",    "Silver", "999 Old Street", date(2021, 1, 1),  date(2022, 3, 14), False),
]

existing_schema = StructType([
    StructField("customer_key",  StringType(), True),
    StructField("customer_id",   StringType(), False),
    StructField("full_name",     StringType(), True),
    StructField("email",         StringType(), True),
    StructField("tier",          StringType(), True),
    StructField("address",       StringType(), True),
    StructField("start_date",    DateType(),   True),
    StructField("end_date",      DateType(),   True),
    StructField("is_current",    BooleanType(),True),
])

existing_df = spark.createDataFrame(existing_data, schema=existing_schema)
print("=== EXISTING DIMENSION TABLE ===")
existing_df.show(truncate=False)



In [0]:
updates_data = [
    # C001 — email changed (Gold customer updated contact)
    ("C001", "Alice Johnson",  "alice.new@email.com", "Gold",     "123 Main St"),
    # C002 — tier upgraded Silver → Gold
    ("C002", "Bob Smith",      "bob@email.com",        "Gold",     "456 Oak Ave"),
    # C003 — no change at all (same values)
    ("C003", "Carol White",    "carol@email.com",      "Bronze",   "789 Pine Rd"),
    # C005 — brand new customer, not in dimension yet
    ("C005", "Emma Davis",     "emma@email.com",       "Silver",   "555 New Blvd"),
]

updates_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("full_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("tier",        StringType(), True),
    StructField("address",     StringType(), True),
])

updates_df = spark.createDataFrame(updates_data, schema=updates_schema)
print("=== UPDATING TABLE ===")
updates_df.show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from datetime import date

def apply_scd2(existing_df, updates_df):
    today = F.lit(str(date.today()))
    COMPARE_COLS = ["full_name","email","tier","address"]

    current = existing_df.filter(F.col("is_current") ==True)

    joined = updates_df.alias("u").join(current.alias("c"), 
                                        F.col("u.customer_id")==F.col("c.customer_id"),
                                        how="left")

    changed = joined.filter(
        (F.col("u.full_name") != F.col("c.full_name")) | 
        (F.col("u.email") != F.col("c.email")) |
        (F.col("u.tier") != F.col("c.tier")) |
        (F.col("u.address") != F.col("c.address")) |
        (F.col("c.customer_id").isNull())
    )

    ids_to_expire = changed.filter(F.col("c.customer_id").isNotNull()).select(F.col("c.customer_id")).alias("customer_id")
    
    expired = current.join(ids_to_expire, on="customer_id",how="inner")\
        .withColumn("end_date", today.cast("date"))\
        .withColumn("is_current", F.lit(False))

    new_rows = changed.select(
        F.expr("uuid()").alias("customer_key"),
        F.col("u.customer_id"),
        F.col("u.full_name"),
        F.col("u.email"),
        F.col("u.tier"),
        F.col("u.address"),
        today.cast("date").alias("start_date"),
        F.lit(None).cast("date").alias("end_date"),
        F.lit(True).alias("is_current")
    )

    not_changed = existing_df.join(ids_to_expire, on="customer_id", how="left_anti")

    final_df = not_changed.unionByName(expired).unionByName(new_rows)
    return final_df

final_df = apply_scd2(existing_df, updates_df)
final_df = final_df.orderBy("customer_id", "start_date")
display(final_df)